# Gabarito — Áreas de Superfícies Planas

Este notebook traz a resolução comentada dos exercícios de [`0420a_areas_de_superficies_planas.ipynb`](0420a_areas_de_superficies_planas.ipynb).

⚠️ **Antes de olhar aqui:** tente resolver os exercícios sozinho no notebook original e use as células de verificação (`assert`) para conferir sua resposta. Volte a este gabarito só depois de tentar — ou se travar de verdade em algum passo.

In [1]:
# Imports e ferramentas necessários para os exercícios abaixo (reconstruídos do notebook principal)
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.patches import Circle, Polygon, Rectangle, Wedge

# Paleta de cores Nord
NORD_FUNDO = "#2E3440"
NORD_PAINEL = "#3B4252"
NORD_LINHA = "#4C566A"
NORD_TEXTO = "#ECEFF4"
NORD_TEXTO_SEC = "#D8DEE9"
COR_PONTO = "#88C0D0"
COR_DESTAQUE = "#A3BE8C"
COR_ALERTA = "#BF616A"
COR_SECUNDARIA = "#81A1C1"
COR_EXTRA = "#B48EAD"
COR_AVISO = "#EBCB8B"


def estilizar_eixo(ax) -> None:
    """Aplica o estilo Nord padrão a um eixo do matplotlib."""
    ax.set_facecolor(NORD_FUNDO)
    ax.tick_params(colors=NORD_TEXTO_SEC)
    for spine in ax.spines.values():
        spine.set_color(NORD_LINHA)
    ax.grid(color=NORD_LINHA, linewidth=0.5, alpha=0.5)

# Ferramentas de desenho e de medida dos notebooks anteriores (0402a, 0407a, 0410a, 0412a, 0418a e 0419a)

def preparar_eixo(ax, pontos, margem: float = 0.8) -> None:
    """Deixa o eixo pronto para desenhos de geometria (estilo Nord, sem grade, sem números,
    escala igual) e enquadra todos os `pontos` com uma margem em volta."""
    estilizar_eixo(ax)
    ax.grid(False)
    xs = [p[0] for p in pontos]
    ys = [p[1] for p in pontos]
    ax.set_xlim(min(xs) - margem, max(xs) + margem)
    ax.set_ylim(min(ys) - margem, max(ys) + margem)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])


def direcao(angulo_graus: float) -> np.ndarray:
    """Vetor de tamanho 1 apontando na direção dada, em graus, a partir da horizontal (sentido anti-horário).

    (cos e sin vêm da Trigonometria, módulo 05; aqui só servem para apontar na direção certa.)
    """
    rad = math.radians(angulo_graus)
    return np.array([math.cos(rad), math.sin(rad)])


def direcao_de(origem, destino) -> float:
    """Direção, em graus, em que `destino` está quando visto de `origem`."""
    return math.degrees(math.atan2(destino[1] - origem[1], destino[0] - origem[0]))


def ponto_medio(P, Q) -> np.ndarray:
    """Ponto médio do segmento PQ: a média das coordenadas (0402a)."""
    return (np.asarray(P, dtype=float) + np.asarray(Q, dtype=float)) / 2


def iguais(x: float, y: float) -> bool:
    """Compara duas medidas com uma tolerância pequena (as contas com float têm arredondamentos)."""
    return math.isclose(x, y, rel_tol=1e-9, abs_tol=1e-6)


def pe_da_altura(X, P, Q) -> np.ndarray:
    """Pé da perpendicular baixada de X sobre a reta PQ (0407a)."""
    X, P, Q = (np.asarray(Z, dtype=float) for Z in (X, P, Q))
    u = Q - P
    return P + np.dot(X - P, u) / np.dot(u, u) * u


def distancia_ponto_reta(X, P, Q) -> float:
    """Distância do ponto X à reta PQ: a medida de X até o pé da perpendicular (0407a)."""
    return math.dist(X, pe_da_altura(X, P, Q))


def incentro(A, B, C) -> np.ndarray:
    """Incentro: média dos vértices com pesos iguais aos lados opostos, (a·A + b·B + c·C)/(a + b + c) (0410a)."""
    A, B, C = (np.asarray(Z, dtype=float) for Z in (A, B, C))
    a, b, c = math.dist(B, C), math.dist(C, A), math.dist(A, B)
    return (a * A + b * B + c * C) / (a + b + c)


def circuncentro(A, B, C) -> np.ndarray:
    """Circuncentro: o ponto equidistante dos três vértices, solução de um sistema linear 2×2 (0410a e 0412a)."""
    A, B, C = (np.asarray(Z, dtype=float) for Z in (A, B, C))
    M = 2 * np.array([B - A, C - A])
    return np.linalg.solve(M, np.array([B @ B - A @ A, C @ C - A @ A]))


def desenhar_ponto(ax, p, nome: str = "", cor: str = COR_AVISO, deslocamento=(0.12, 0.15),
                   tamanho_fonte: int = 13) -> None:
    """Desenha um ponto com seu nome (letra maiúscula) ao lado."""
    ax.plot(*p, "o", color=cor, markersize=7, zorder=8)
    if nome:
        ax.text(p[0] + deslocamento[0], p[1] + deslocamento[1], nome, color=NORD_TEXTO, fontsize=tamanho_fonte,
                fontweight="bold", zorder=9)


def desenhar_segmento(ax, P, Q, cor: str = NORD_TEXTO, largura: float = 3.0, estilo: str = "-",
                      alpha: float = 1.0, zorder: int = 6) -> None:
    """Desenha o segmento PQ (só o pedaço entre P e Q)."""
    ax.plot(*zip(P, Q), color=cor, lw=largura, linestyle=estilo, alpha=alpha, zorder=zorder,
            solid_capstyle="round")


def rotular_segmento(ax, P, Q, texto: str, cor: str = NORD_TEXTO, distancia: float = 0.35, longe_de=None,
                     tamanho: int = 11) -> None:
    """Escreve `texto` ao lado do meio do segmento PQ, afastado `distancia` na direção perpendicular a ele.
    Se `longe_de` for um ponto, o texto vai para o lado do segmento oposto a esse ponto."""
    P, Q = np.asarray(P, dtype=float), np.asarray(Q, dtype=float)
    u = (Q - P) / np.linalg.norm(Q - P)
    normal = np.array([-u[1], u[0]])
    meio = (P + Q) / 2
    if longe_de is not None and np.dot(meio - np.asarray(longe_de, dtype=float), normal) < 0:
        normal = -normal
    ax.text(*(meio + distancia * normal), texto, color=cor, fontsize=tamanho, fontweight="bold",
            ha="center", va="center", zorder=9)


def escrever_checklist(ax, itens, titulo: str = "") -> None:
    """Painel de texto com uma linha por item (texto, ok): ✓ verde quando vale, ✗ vermelho quando não (0410a)."""
    ax.set_facecolor(NORD_FUNDO)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    if titulo:
        ax.text(0.0, 0.97, titulo, color=NORD_TEXTO, fontsize=12, fontweight="bold", va="top")
    passo = min(0.13, 0.85 / max(len(itens), 1))
    for k, (texto, ok) in enumerate(itens):
        ax.text(0.0, 0.84 - k * passo, f"{'✓' if ok else '✗'}  {texto}", color=COR_DESTAQUE if ok else COR_ALERTA,
                fontsize=11, va="top")


def desenhar_circunferencia(ax, O, r: float, cor: str = COR_PONTO, estilo: str = "-",
                            largura: float = 2.5, preencher: bool = False, opacidade: float = 0.15) -> None:
    """Desenha a circunferência de centro O e raio r (a linha). Com `preencher=True`, pinta também a
    região interna: o círculo (0412a)."""
    O = tuple(np.asarray(O, dtype=float))
    if preencher:
        ax.add_patch(Circle(O, r, facecolor=cor, alpha=opacidade, edgecolor="none", zorder=1))
    ax.add_patch(Circle(O, r, fill=False, edgecolor=cor, lw=largura, linestyle=estilo, zorder=3))


def vertices_regular(n: int, R: float = 1.0, centro=(0.0, 0.0), rotacao: float | None = None) -> np.ndarray:
    """Vértices do polígono regular de n lados inscrito na circunferência λ(centro, R): n pontos dela separados
    por 360°/n, o primeiro na direção `rotacao` (em graus). Sem `rotacao`, o polígono fica "apoiado" num lado
    horizontal (0418a).

    (A direção de cada vértice usa `direcao`, que vem da Trigonometria; aqui ela só aponta os vértices.)
    """
    inicio = -90 - 180 / n if rotacao is None else rotacao
    c = np.asarray(centro, dtype=float)
    return np.array([c + R * direcao(inicio + 360 * k / n) for k in range(n)])


def area_cadarco(pontos) -> float:
    """Área de um polígono simples (lados que não se cruzam) a partir dos vértices EM ORDEM, pela fórmula do
    cadarço: S = |x₁y₂ − x₂y₁ + x₂y₃ − x₃y₂ + ... + xₙy₁ − x₁yₙ| / 2 (a "régua de área" de 0419a).

    A seção 3 deste notebook mostra de onde ela vem no caso do triângulo; a demonstração geral chega em
    Geometria Analítica (módulo 07).
    """
    P = np.asarray(pontos, dtype=float)
    x, y = P[:, 0], P[:, 1]
    return abs(float(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))) / 2


def desenhar_peca(ax, pontos, cor: str = COR_PONTO, opacidade: float = 0.55, borda: str = NORD_TEXTO,
                  largura: float = 1.5, estilo: str = "-", rotulo: str | None = None, tamanho: int = 11,
                  zorder: int = 3) -> None:
    """Desenha uma região (polígono) pintada, com contorno; `rotulo` é escrito no centro dela (0419a)."""
    P = np.asarray(pontos, dtype=float)
    ax.add_patch(Polygon(P, closed=True, facecolor=cor, alpha=opacidade, edgecolor="none", zorder=zorder))
    ax.add_patch(Polygon(P, closed=True, fill=False, edgecolor=borda, lw=largura, linestyle=estilo,
                         zorder=zorder + 1))
    if rotulo:
        ax.text(*P.mean(axis=0), rotulo, color=NORD_TEXTO, fontsize=tamanho, fontweight="bold", ha="center",
                va="center", zorder=zorder + 2)


# Ferramentas novas deste notebook
# Cores fixas: círculo azul-claro; setor amarelo; triângulo central roxo; segmento circular verde; coroa laranja;
# parte que é subtraída vermelha; polígono que aproxima uma curva azul.
COR_CIRCULO = COR_PONTO  # círculo: azul-claro
COR_SETOR = COR_AVISO  # setor circular: amarelo
COR_TRIANGULO = COR_EXTRA  # triângulo central (dois raios e a corda): roxo
COR_SEGMENTO = COR_DESTAQUE  # segmento circular: verde
COR_COROA = "#D08770"  # coroa circular: laranja
COR_SUBTRAIDA = COR_ALERTA  # parte que é retirada (ou erro): vermelho
COR_POLIGONO = COR_SECUNDARIA  # polígono que aproxima a curva: azul
CORES_PECAS = ("#88C0D0", "#A3BE8C", "#EBCB8B", "#BF616A", "#B48EAD", "#D08770", "#81A1C1", "#8FBCBB", "#5E81AC")


def ponto_na_circunferencia(O, r: float, angulo_graus: float) -> np.ndarray:
    """O ponto da circunferência λ(O, r) na direção `angulo_graus`, vista a partir do centro (0412a)."""
    return np.asarray(O, dtype=float) + r * direcao(angulo_graus)


def pontos_arco(O, r: float, inicio: float, fim: float, n: int = 200) -> np.ndarray:
    """n + 1 pontos do arco de λ(O, r) que vai da direção `inicio` até a direção `fim` (em graus; se fim < inicio,
    o arco é percorrido no sentido horário). Ligados por segmentos, eles formam uma poligonal "colada" no arco:
    é assim que este notebook DESENHA as regiões curvas e as MEDE com a régua de área (area_cadarco)."""
    angulos = np.radians(np.linspace(inicio, fim, n + 1))
    return np.asarray(O, dtype=float) + r * np.column_stack([np.cos(angulos), np.sin(angulos)])


def regiao_circulo(O, r: float, n: int = 400) -> np.ndarray:
    """O círculo λ(O, r) aproximado por um polígono regular de n lados (para desenhar e para medir)."""
    return pontos_arco(O, r, 0, 360, n)[:-1]


def escrever_area(ax, P, texto: str, cor: str = NORD_TEXTO, tamanho: int = 12) -> None:
    """Escreve um rótulo de área em P, com uma "etiqueta" escura atrás para ficar legível sobre as cores."""
    ax.text(*P, texto, color=cor, fontsize=tamanho, fontweight="bold", ha="center", va="center", zorder=12,
            bbox={"boxstyle": "round,pad=0.25", "facecolor": NORD_FUNDO, "edgecolor": "none", "alpha": 0.75})


# Funções construídas nas seções 1 e 5 do notebook principal

def area_circulo(r: float) -> float:
    """Área do círculo de raio r: S = π·r²."""
    return math.pi * r**2


def area_setor(r: float, alfa: float) -> float:
    """Área do setor de α graus: (α/360°)·πr²."""
    return alfa / 360 * area_circulo(r)


def dobrar_lado(lado: float, R: float = 1.0) -> float:
    """Lado ℓ₂ₙ do regular de 2n lados inscrito em λ(O, R), a partir do lado ℓₙ (Pitágoras duas vezes, 0418a)."""
    return math.sqrt(2 * R**2 - R * math.sqrt(4 * R**2 - lado**2))


def area_regular_inscrito(n: int, R: float = 1.0) -> float:
    """Área S = p·a do regular de n = 6·2ᵏ lados inscrito em λ(O, R), só com Pitágoras: começa no hexágono
    (lado = R) e dobra o número de lados até chegar a n."""
    lados, lado = 6, R
    while lados < n:
        lado, lados = dobrar_lado(lado, R), 2 * lados
    if lados != n:
        raise ValueError("use n = 6, 12, 24, 48, ... (o hexágono dobrado várias vezes)")
    apotema = math.sqrt(R**2 - (lado / 2) ** 2)
    return (n * lado / 2) * apotema


def lado_e_apotema(n: int, R: float) -> tuple[float, float]:
    """Lado e apótema do regular de n lados inscrito em λ(O, R), MEDIDOS nas coordenadas dos vértices (0418a)."""
    V = vertices_regular(n, R)
    return math.dist(V[0], V[1]), math.dist((0, 0), ponto_medio(V[0], V[1]))


def area_regular(n: int, R: float = 1.0) -> float:
    """Área S = p·a do regular de n lados inscrito em λ(O, R) (0418a)."""
    lado, apotema = lado_e_apotema(n, R)
    return n * lado / 2 * apotema


def monte_carlo_area(dentro, caixa: tuple[float, float, float, float], n: int, semente: int = 0) -> float:
    """Estima a área da região {(x, y): dentro(x, y)} sorteando n pontos uniformes na caixa
    (xmin, xmax, ymin, ymax): área ≈ (fração dos pontos que caem dentro) × área da caixa.
    `dentro` recebe dois arrays numpy x e y e devolve um array de True/False."""
    xmin, xmax, ymin, ymax = caixa
    gerador = np.random.default_rng(semente)
    x, y = gerador.uniform(xmin, xmax, n), gerador.uniform(ymin, ymax, n)
    return float(np.mean(dentro(x, y))) * (xmax - xmin) * (ymax - ymin)


def dentro_circulo(x, y):
    return x**2 + y**2 <= 1

## Básico

**Exercício 1.** Círculos, setores, segmentos e coroas.

In [2]:
# (a) Diâmetro 12 ⇒ raio 6: π·6² = 36π (ou, direto pelo diâmetro, π·12²/4)
area_ex1a = math.pi * 12**2 / 4

# (b) 40° é 40/360 = 1/9 do círculo, cuja área é 81π: o setor vale 9π
area_ex1b = 40 / 360 * math.pi * 9**2

# (c) Setor de 90°: um quarto de 16π = 4π. Os dois raios são perpendiculares, então o triângulo central é
#     retângulo de catetos 4 e 4: área 8. Segmento = 4π − 8
area_ex1c = 90 / 360 * math.pi * 4**2 - 4 * 4 / 2

# (d) Setor de 120°: um terço de 100π. A corda mede 10√3; a distância do centro à corda sai de Pitágoras,
#     d = √(10² − (5√3)²) = √25 = 5. Triângulo central: 10√3 · 5 / 2 = 25√3
corda_d = 10 * math.sqrt(3)
d_d = math.sqrt(10**2 - (corda_d / 2) ** 2)
area_ex1d = 120 / 360 * math.pi * 10**2 - corda_d * d_d / 2

# (e) Coroa: π(7² − 3²) = 40π
area_ex1e = math.pi * (7**2 - 3**2)

# (f) 240° > 180°: o centro fica DENTRO do segmento, e o triângulo é somado. A corda é a mesma do item (d):
#     os arcos de 120° e de 240° têm as mesmas pontas
area_ex1f = 240 / 360 * math.pi * 10**2 + corda_d * d_d / 2

print(area_ex1a, area_ex1b, area_ex1c, area_ex1d, area_ex1e, area_ex1f)
assert math.isclose(area_ex1d + area_ex1f, math.pi * 10**2)  # os dois segmentos somam o círculo

113.09733552923255 28.274333882308138 4.5663706143591725 61.41848493043782 125.66370614359172 252.74078042854148


A última linha confere os itens (d) e (f) de uma vez: a mesma corda divide o círculo em dois segmentos, de 120° e de 240°, e eles somam $100\pi$. No item (a), o erro clássico é escrever $\pi \cdot 12^2 = 144\pi$, quatro vezes a resposta certa.

**Exercício 2.** Qual fórmula usar?

In [3]:
# (a) Três lados ⇒ Herão, com p = (7 + 8 + 9)/2 = 12: √(12·5·4·3) = √720 = 12√5
p = (7 + 8 + 9) / 2
metodo_ex2a, area_ex2a = "herao", math.sqrt(p * (p - 7) * (p - 8) * (p - 9))

# (b) Vértices ⇒ coordenadas: ½|1·(2 − 7) + 6·(7 − 1) + 3·(1 − 2)| = ½|−5 + 36 − 3| = 14
metodo_ex2b, area_ex2b = "coordenadas", abs(1 * (2 - 7) + 6 * (7 - 1) + 3 * (1 - 2)) / 2

# (c) Catetos ⇒ b·c/2 = 54
metodo_ex2c, area_ex2c = "catetos", 9 * 12 / 2

# (d) Polígono regular ⇒ p·a (a função area_regular da seção 1); confere com 2√2·R² = 50√2
metodo_ex2d, area_ex2d = "p_a", area_regular(8, 5)

# (e) Perímetro e raio inscrito ⇒ p·r, com o SEMIperímetro 15: 15 · 2,5 = 37,5
metodo_ex2e, area_ex2e = "p_r", (30 / 2) * 2.5

print(metodo_ex2a, area_ex2a, metodo_ex2b, area_ex2b, metodo_ex2c, area_ex2c)
print(metodo_ex2d, area_ex2d, metodo_ex2e, area_ex2e)

herao 26.832815729997478 coordenadas 14.0 catetos 54.0
p_a 70.71067811865476 p_r 37.5


Em todos os itens daria para chegar à área por outros caminhos (no item (a), achando a altura por Pitágoras; no item (c), por Herão com a hipotenusa 15), mas o caminho escolhido é o que usa **diretamente** os dados do enunciado, sem passos intermediários. No item (e), usar 30 no lugar de 15 dobraria a área: é o erro "perímetro no lugar do semiperímetro".

**Exercício 3.** Unidades e escala.

In [4]:
# (a) 1 m² = 100 cm × 100 cm = 10 000 cm²
cm2_ex3a = 3.5 * 100**2

# (b) 250 × 400 = 100 000 m²; 1 ha = 10 000 m² e 1 km² = 1 000 000 m²
area_m2 = 250 * 400
ha_ex3b, km2_ex3b = area_m2 / 10_000, area_m2 / 1_000_000

# (c) Escala 1 : 200: 2,5 cm → 500 cm = 5 m e 3 cm → 600 cm = 6 m; área real 30 m²
#     (pela regra: 7,5 cm² × 200² = 300 000 cm² = 30 m²)
m2_ex3c = (2.5 * 200 / 100) * (3 * 200 / 100)

# (d) 8 cm² × 50 000² = 2·10¹⁰ cm² = 2·10⁶ m² = 2 km² = 200 ha
area_cm2 = 8 * 50_000**2
km2_ex3d, ha_ex3d = area_cm2 / 10_000 / 1_000_000, area_cm2 / 10_000 / 10_000

# (e) 10 alqueires paulistas = 242 000 m² = 24,2 ha
ha_ex3e = 10 * 24_200 / 10_000

print(cm2_ex3a, ha_ex3b, km2_ex3b, m2_ex3c, km2_ex3d, ha_ex3d, ha_ex3e)

35000.0 10.0 0.1 30.0 2.0 200.0 24.2


O item (d) mostra por que mapas de escala pequena enganam: 8 cm² parecem nada, mas, com a escala $1 : 50\,000$, cada cm² do papel vale $50\,000^2 = 2{,}5$ bilhões de cm² reais, isto é, $0{,}25$ km². No item (e), com alqueires **mineiros** a fazenda teria o dobro: 48,4 ha.

## Intermediário

**Exercício 4.** As suas próprias funções.

In [5]:
def area_circulo_ex4(r: float) -> float:
    return math.pi * r**2


def area_setor_ex4(r: float, alfa: float) -> float:
    # A fração α/360° do círculo
    return alfa / 360 * area_circulo_ex4(r)


def area_segmento_ex4(r: float, alfa: float) -> float:
    # Corda MEDIDA entre as pontas do arco; d por Pitágoras; triângulo = corda·d/2
    A = ponto_na_circunferencia((0, 0), r, 0)
    B = ponto_na_circunferencia((0, 0), r, alfa)
    corda = math.dist(A, B)
    d = math.sqrt(max(r**2 - (corda / 2) ** 2, 0.0))  # max(..., 0) evita raiz de −0,0000000001 em α = 180°
    triangulo = corda * d / 2
    if alfa < 180:
        return area_setor_ex4(r, alfa) - triangulo
    return area_setor_ex4(r, alfa) + triangulo  # α ≥ 180°: o centro está dentro do segmento


def area_coroa_ex4(R: float, r: float) -> float:
    return math.pi * (R**2 - r**2)


def area_triangulo_coord_ex4(A, B, C) -> float:
    (x1, y1), (x2, y2), (x3, y3) = A, B, C
    return abs(x1 * (y2 - y3) + x2 * (y3 - y1) + x3 * (y1 - y2)) / 2


def area_poligono_ex4(vertices) -> float:
    V = [tuple(v) for v in vertices]

    def orientada(P, Q, R):
        # Mesma expressão do triângulo, SEM o módulo: o sinal diz o sentido de giro
        return (P[0] * (Q[1] - R[1]) + Q[0] * (R[1] - P[1]) + R[0] * (P[1] - Q[1])) / 2

    return abs(sum(orientada(V[0], V[k], V[k + 1]) for k in range(1, len(V) - 1)))


print(area_circulo_ex4(1), area_setor_ex4(6, 60), area_segmento_ex4(6, 60), area_coroa_ex4(5, 3))
print(area_triangulo_coord_ex4((0, 0), (4, 1), (1, 3)), area_poligono_ex4([(0, 0), (6, 3), (0, 6), (2, 3)]))

3.141592653589793 18.84955592153876 3.2610986534188626 50.26548245743669
5.5 12.0


Dois detalhes técnicos. Em $\alpha = 180°$, a corda é o diâmetro e $r^2 - (\text{corda}/2)^2$ deveria ser 0, mas as contas com `float` podem dar um número negativo minúsculo, e `math.sqrt` reclamaria; o `max(..., 0.0)` resolve. Em `area_poligono_ex4`, o módulo é tirado **só no final**: se fosse tirado em cada triângulo, o polígono côncavo da "ponta de flecha" daria $18 + 6 = 24$ em vez de $18 - 6 = 12$.

**Exercício 5.** Regiões compostas.

In [6]:
# (a) Pista = retângulo 100 × 60 + dois semicírculos de raio 30 (= um círculo inteiro de raio 30).
#     Contorno = duas retas de 100 m + duas semicircunferências (= uma circunferência de raio 30)
area_ex5a = 100 * 60 + math.pi * 30**2
perimetro_ex5a = 2 * 100 + 2 * math.pi * 30

# (b) Subtração: quadrado − círculo
area_ex5b = 20**2 - math.pi * 6**2

# (c) Subtração: os quatro quartos de círculo de raio 5 formam um círculo inteiro de raio 5
area_ex5c = 10**2 - math.pi * 5**2

# (d) Hexágono de lado 4: seis equiláteros de lado 4. O raio da circunscrita é o lado (4) e o da inscrita
#     é o apótema, √(4² − 2²) = 2√3 (Pitágoras na metade de um equilátero)
hexagono_ex5d = 6 * 4**2 * math.sqrt(3) / 4
circunscrito_ex5d = math.pi * 4**2
apotema = math.sqrt(4**2 - 2**2)
inscrito_ex5d = math.pi * apotema**2
razao_ex5d = inscrito_ex5d / circunscrito_ex5d  # = (a/R)² = 3/4


# (e) Pétala: dois quartos de círculo = quadrado + pétala (a pétala foi coberta duas vezes)
def area_petala_ex5(l: float) -> float:
    return 2 * (math.pi * l**2 / 4) - l**2


# Reuleaux: três setores de 60° cobrem o triângulo três vezes; tira-se o triângulo duas vezes
def area_reuleaux_ex5(l: float) -> float:
    return 3 * (math.pi * l**2 / 6) - 2 * (l**2 * math.sqrt(3) / 4)


print(area_ex5a, perimetro_ex5a, area_ex5b, area_ex5c)
print(hexagono_ex5d, circunscrito_ex5d, inscrito_ex5d, razao_ex5d, area_petala_ex5(1), area_reuleaux_ex5(1))

8827.433388230813 388.4955592153876 286.90266447076743 21.46018366025517
41.569219381653056 50.26548245743669 37.69911184307751 0.7499999999999999 0.5707963267948966 0.704770923010458


No item (a), a armadilha é o perímetro: os dois lados de 60 m do retângulo ficam **por dentro** da pista (são as "emendas" com os semicírculos) e não fazem parte do contorno. No item (d), a razão $\frac{3}{4}$ sai sem calcular nenhuma área: os dois círculos são semelhantes, com razão $\frac{a}{R} = \frac{\sqrt{3}}{2}$, e as áreas estão na razão $k^2 = \frac{3}{4}$. E o hexágono ($\approx 41{,}57$) fica entre os dois círculos ($\approx 37{,}70$ e $\approx 50{,}27$), como deveria.

**Exercício 6.** Problemas de dois ou três passos.

In [7]:
# (a) Anilha: R² − r² = (c/2)² = 144, então a coroa vale π·144, sem precisar de R nem de r.
#     Com r = 5: R = √(25 + 144) = 13
c = 24
area_ex6a = math.pi * c**2 / 4
R_ex6a = math.sqrt(5**2 + (c / 2) ** 2)

# (b) S = p·r ⇒ r = S/p. Triângulo 13–14–15: S = 84, p = 21, r = 4 ⇒ fração 16π/84.
#     Triângulo 3–4–5: S = 6, p = 6, r = 1 ⇒ fração π/6
S, p = 84, 21
r = S / p
fracao_ex6b = math.pi * r**2 / S
fracao_345_ex6b = math.pi * (6 / 6) ** 2 / 6

# (c) Hexágono de lado 3: seis equiláteros, 6·9√3/4 = 13,5√3 ≈ 23,38 m²; 10% a mais; R$ 60,00 o m²
area_ex6c = 6 * 3**2 * math.sqrt(3) / 4
comprar_ex6c = area_ex6c * 1.10
custo_ex6c = comprar_ex6c * 60

print(area_ex6a, R_ex6a, fracao_ex6b, fracao_345_ex6b, area_ex6c, comprar_ex6c, custo_ex6c)

452.3893421169302 13.0 0.5983986006837702 0.5235987755982988 23.38268590217984 25.720954492397826 1543.2572695438696


No item (b), o círculo inscrito ocupa cerca de 60% do triângulo 13–14–15 e 52% do 3–4–5. Para o triângulo **equilátero**, a fração é $\frac{\pi}{3\sqrt{3}} \approx 0{,}605$, e esse é o máximo possível: nenhum triângulo "aproveita" melhor o círculo inscrito que o equilátero (mais um parente da isoperimetria do Desafio 8). No item (c), compram-se cerca de $25{,}72$ m² de piso, por cerca de R\$ 1.543,26.

## Desafio

**Exercício 7.** O Teorema de Pick.

In [9]:
def pick_ex7(vertices) -> tuple[int, int]:
    V = [(int(x), int(y)) for x, y in vertices]
    lados = list(zip(V, V[1:] + V[:1]))

    # B: num lado, os pontos inteiros dividem o segmento em mdc(|Δx|, |Δy|) partes iguais
    B = sum(math.gcd(abs(q[0] - p[0]), abs(q[1] - p[1])) for p, q in lados)

    def na_borda(x: int, y: int) -> bool:
        # Colinear com o lado (produto vetorial zero) e dentro do retângulo do lado
        for (x1, y1), (x2, y2) in lados:
            if (x2 - x1) * (y - y1) - (y2 - y1) * (x - x1) == 0 \
                    and min(x1, x2) <= x <= max(x1, x2) and min(y1, y2) <= y <= max(y1, y2):
                return True
        return False

    def dentro(x: int, y: int) -> bool:
        # Semirreta horizontal para a direita: conta os lados cruzados (ímpar = dentro).
        # A regra (y1 > y) != (y2 > y) conta cada vértice uma vez só, mesmo com lados horizontais
        cruzamentos = 0
        for (x1, y1), (x2, y2) in lados:
            if (y1 > y) != (y2 > y) and x1 + (y - y1) * (x2 - x1) / (y2 - y1) > x:
                cruzamentos += 1
        return cruzamentos % 2 == 1

    xs, ys = [x for x, _ in V], [y for _, y in V]
    I = sum(1 for x in range(min(xs), max(xs) + 1) for y in range(min(ys), max(ys) + 1)
            if not na_borda(x, y) and dentro(x, y))
    return I, B


print(pick_ex7([(0, 0), (4, 1), (1, 3)]), pick_ex7([(0, 0), (4, 0), (4, 4), (0, 4)]))
I, B = pick_ex7([(0, 0), (4, 1), (1, 3)])
print(f"triângulo (0, 0), (4, 1), (1, 3): I + B/2 − 1 = {I} + {B}/2 − 1 = {I + B / 2 - 1}  (área: 5,5)")

(5, 3) (9, 16)
triângulo (0, 0), (4, 1), (1, 3): I + B/2 − 1 = 5 + 3/2 − 1 = 5.5  (área: 5,5)


O teste do ponto **na borda** vem antes do teste "dentro": os pontos da borda são exatamente os casos em que a semirreta passa por um vértice ou corre ao longo de um lado, e o critério da paridade fica ambíguo. Separando-os primeiro, a contagem fica limpa.

O Teorema de Pick é surpreendente: a área, uma grandeza contínua, sai de **contar pontos**. Ele só vale para polígonos com vértices em pontos inteiros (e sem buracos). Uma consequência curiosa: todo polígono desse tipo tem área múltipla de $\frac{1}{2}$, e por isso **não existe** triângulo equilátero com os três vértices em pontos inteiros (a área dele teria um $\sqrt{3}$).

**Exercício 8.** Isoperimetria numérica.

In [10]:
# (a) Perímetro 12: equilátero de lado 4, quadrado de lado 3, hexágono de lado 2, círculo com 2πr = 12
areas_ex8 = {
    3: 4**2 * math.sqrt(3) / 4,
    4: 3**2,
    6: 6 * 2**2 * math.sqrt(3) / 4,
    "círculo": math.pi * (12 / (2 * math.pi)) ** 2,  # = 36/π
}

# (b) O regular inscrito no raio 1, ampliado na razão k = 12/perímetro: a área vai × k²
lista_ex8 = []
for n in range(3, 41):
    V = vertices_regular(n, 1)
    perimetro = n * math.dist(V[0], V[1])
    k = 12 / perimetro
    lista_ex8.append(area_cadarco(V) * k**2)

# (c) Todas as áreas ficam abaixo da do círculo
conclusao_ex8 = "círculo" if areas_ex8["círculo"] > max(lista_ex8) else "quadrado"

print(areas_ex8)
print([round(s, 4) for s in lista_ex8[:6]], "...", round(lista_ex8[-1], 4))

{3: 6.928203230275509, 4: 9, 6: 10.392304845413264, 'círculo': 11.459155902616464}
[6.9282, 9.0, 9.9099, 10.3923, 10.6793, 10.864] ... 11.4356


As áreas sobem com o número de lados: $6{,}93$ (triângulo), $9$ (quadrado), $\approx 9{,}91$ (pentágono), $\approx 10{,}39$ (hexágono)... e o polígono de 40 lados chega a $\approx 11{,}436$, já pertinho de $\frac{36}{\pi} \approx 11{,}459$. É a **desigualdade isoperimétrica**: entre todas as figuras planas de mesmo perímetro, o círculo é a de maior área. Aqui ela foi só **ilustrada** com regulares; a demonstração geral é bem mais difícil (e não cabe neste módulo). A natureza "conhece" esse resultado: bolhas de sabão e gotas tendem ao formato redondo.

**Exercício 9.** Monte Carlo × polígonos.

In [11]:
ns_ex9 = [6, 24, 96, 384, 1536, 6144]

# (a) Polígonos inscritos (dobras do hexágono, só Pitágoras)
erros_poligono_ex9 = [math.pi - area_regular_inscrito(n) for n in ns_ex9]

# (b) Monte Carlo com o mesmo número de pontos (semente 0)
erros_mc_ex9 = [abs(monte_carlo_area(dentro_circulo, (-1, 1, -1, 1), n, semente=0) - math.pi) for n in ns_ex9]

# (c) n × 4 são duas dobras; cada dobra divide o erro por ~4, então o erro cai ~16 vezes
razao_ex9 = erros_poligono_ex9[0] / erros_poligono_ex9[1]

# (d) Com n = 6144
vencedor_ex9 = "polígonos" if erros_poligono_ex9[-1] < erros_mc_ex9[-1] else "monte carlo"

display(pd.DataFrame({"n": ns_ex9, "erro do polígono": erros_poligono_ex9, "erro do Monte Carlo": erros_mc_ex9}))
print(f"razão: {razao_ex9:.2f}   vencedor: {vencedor_ex9}")

,n,erro do polígono,erro do Monte Carlo
0,6,5.435164e-01,0.474926
1,24,3.576411e-02,0.025074
2,96,2.242451e-03,0.099926
3,384,1.401813e-04,0.004241
4,1536,8.761433e-06,0.016593
5,6144,5.476945e-07,0.008147


razão: 15.20   vencedor: polígonos


Com 6 144 "contas", o polígono erra só na sétima casa decimal ($\approx 5 \times 10^{-7}$); o Monte Carlo erra por quase $0{,}01$ ($\pi \approx 3{,}133$). Cada vez que $n$ é multiplicado por 4, o erro do polígono cai cerca de 16 vezes (é $\propto \frac{1}{n^2}$), e o do Monte Carlo cai só cerca de 2 vezes ($\propto \frac{1}{\sqrt{n}}$), e ainda com sobe-e-desce, porque é aleatório.

Então por que alguém usaria Monte Carlo? Porque ele **não precisa de nenhuma fórmula nem de nenhuma estrutura** da região: basta saber dizer se um ponto está dentro ou fora. Para regiões complicadas e, principalmente, em **muitas dimensões** (onde "polígonos" ficam inviáveis), o Monte Carlo é muitas vezes o único método prático.

**Exercício 10.** As lúnulas, uma por uma.

In [12]:
def lunulas_ex10(b: float, c: float, n: int = 2000) -> tuple[float, float]:
    A, B, C = np.array([0.0, 0.0]), np.array([c, 0.0]), np.array([0.0, b])
    M, raio = ponto_medio(B, C), math.hypot(b, c) / 2  # circunferência de diâmetro BC (passa por A)

    # Lúnula sobre AC (vertical): semicírculo pela esquerda, de C (direção 90°) até A (270°); volta de A até C
    # pelo arco grande, que passa pela direção 180° vista de M
    pequeno = pontos_arco(ponto_medio(A, C), b / 2, 90, 270, n)
    grande = pontos_arco(M, raio, direcao_de(M, A) % 360, direcao_de(M, C) % 360, n)
    lunula_b = area_cadarco(np.vstack([pequeno, grande]))

    # Lúnula sobre AB (horizontal): semicírculo por baixo, de A (180°) até B (360°); volta de B até A pelo arco
    # grande, que passa pela direção −90° vista de M
    pequeno = pontos_arco(ponto_medio(A, B), c / 2, 180, 360, n)
    grande = pontos_arco(M, raio, direcao_de(M, B), direcao_de(M, A), n)
    lunula_c = area_cadarco(np.vstack([pequeno, grande]))
    return lunula_b, lunula_c


for b, c in ((3, 4), (1, 1), (5, 12)):
    lb, lc = lunulas_ex10(b, c)
    print(f"catetos {b} e {c}: lúnulas {lb:.6f} + {lc:.6f} = {lb + lc:.6f}   (b·c/2 = {b * c / 2:g})")

catetos 3 e 4: lúnulas 2.512409 + 3.487588 = 5.999997   (b·c/2 = 6)
catetos 1 e 1: lúnulas 0.250000 + 0.250000 = 0.500000   (b·c/2 = 0.5)
catetos 5 e 12: lúnulas 8.137549 + 21.862436 = 29.999985   (b·c/2 = 30)


No triângulo 3–4–5, as lúnulas valem cerca de $2{,}51$ e $3{,}49$: números sem nada de especial, cada um com $\pi$ "escondido" dentro. Mas a soma dá $6{,}000000$, a área do triângulo. O $\pi$ de uma lúnula cancela o $\pi$ da outra, exatamente como a conta da seção 4 previa: $\frac{\pi}{8}(b^2 + c^2 - a^2) = 0$.

Repare também no cuidado com as direções: o arco grande que fecha cada lúnula é o que fica do lado **de fora** do triângulo (passando pela direção $180°$ ou $-90°$, vista do centro $M$). Se o arco fosse percorrido "pelo outro lado", o polígono atravessaria o triângulo e a área sairia errada.